# Python → Rust Code Migration Evaluator

## Business Context

Organizations often prototype in Python but migrate to Rust or C++ for:

- Performance
- Memory safety
- Production reliability

Before committing engineering resources, companies test:

- Code translation quality
- Latency
- Token cost
- Output readability

## What This Notebook Does

1. Accepts Python code.
2. Converts it to Rust using GPT-4.
3. Evaluates performance and readability.
4. Measures latency + cost.
5. Returns structured metrics.

This simulates a lightweight AI-assisted migration assessment tool.

In [1]:
!pip install openai python-dotenv --quiet
import os
import time
from dotenv import load_dotenv
from openai import OpenAI


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4"

In [7]:
def estimate_cost(tokens, price_per_1k=0.03):
    return round((tokens / 1000) * price_per_1k, 6)

In [8]:
def convert_to_rust(python_code: str):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a senior Rust engineer focused on performance and clarity."
            },
            {
                "role": "user",
                "content": f"Convert this Python code to optimized Rust:\n\n{python_code}"
            }
        ]
    )
    return response

In [4]:
def evaluate_rust_code(rust_code: str):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"""
Evaluate this Rust code for:

1. Performance optimization
2. Readability
3. Idiomatic Rust usage

Provide a concise assessment.

{rust_code}
"""
            }
        ]
    )
    return response

In [9]:
def run_migration(python_code: str):
    start_time = time.time()

    # Convert
    rust_response = convert_to_rust(python_code)
    rust_code = rust_response.choices[0].message.content

    # Evaluate
    evaluation_response = evaluate_rust_code(rust_code)
    evaluation_text = evaluation_response.choices[0].message.content

    total_tokens = (
        rust_response.usage.total_tokens +
        evaluation_response.usage.total_tokens
    )

    latency = round(time.time() - start_time, 2)
    cost_estimate = estimate_cost(total_tokens)

    return {
        "rust_code": rust_code,
        "evaluation": evaluation_text,
        "latency_seconds": latency,
        "tokens_used": total_tokens,
        "estimated_cost_usd": cost_estimate
    }

Test the pipeline

In [10]:
sample_python_code = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""

results = run_migration(sample_python_code)
results

{'rust_code': 'The Python code you gave is a recursive implementation of the Fibonacci sequence. Although recursion is nice and simple, it has a really bad performance (exponential) only because it does calculation same thing over and over again. Therefore, this Python code is not optimal.\n\nHere\'s an optimized Rust implementation of Fibonacci sequence using dynamic programming which has a linear runtime complexity (O(n)):\n\n```rust\nfn fibonacci(n: u32) -> u32 {\n    let mut f = vec![0, 1];\n    for i in 2..=n as usize {\n        f.push(f[i - 1] + f[i - 2]);\n    }\n    f[n as usize]\n}\n\nfn main() {\n    let n = 10;      // Change this value to calculate another term\n    println!("{}", fibonacci(n));\n}\n```\n\nThis code defines a function `fibonacci` that takes an unsigned 32-bit integer `n` and returns the nth term of the Fibonacci sequence. In the `fibonacci` function, we maintain a dynamic programming array `f` and use iteration to fill up the array. Finally, we return the n